# ETL — VISp Inhibitory Patch-seq: Cluster Membership & Cell-to-Cluster Mapping

For VISp inhibitory Patch-seq cells (`project_id="visp_patchseq"`, `dataset_id="visp_inh_patchseq"`),
this notebook registers two assignments per cell:

1. **T-type → Tasic 2018 VISp scRNA-seq taxonomy** as `CellToClusterMapping` (the cells were
   not part of Tasic — this is a *mapping*). Source: `patchseq_tx_cell_ttype_labels.csv`,
   column `ttype`, indexed by cell id. **No `ET → PT` translation** (that was an exc-only
   convention; inhibitory ttypes don't contain `ET`).
2. **MET-type → VISp MET-types taxonomy** as `ClusterMembership` (these cells *belong* to
   these MET-types by direct measurement — same cohort that defined the taxonomy). Source:
   `visp_met_cell_assignments_text_names.csv`, column `met_type`, indexed by cell id.

Both writes use **parent propagation**: one row per (cell, ancestor) pair walked from the
leaf to the root via `walk_ancestors` (in `connects_common_connectivity.io.write_utils`).
`probability` is left null (no probability column in either source).

## Section 0 — register missing inhibitory dataset associations

The MET CSV has 495 cells. All 495 are already in `dataitem/` (registered by earlier
notebooks), but only 392 are associated with `dataset_id="visp_inh_patchseq"`. The other
103 are GABAergic MET-types with no dataset association at all. Section 0 appends the
missing 103 `DataItemDataSetAssociation` rows so every MET cell has the proper inh
dataset link before membership is written. (This mirrors the pattern in
`etl_visp_inh_patchseq_02_cell_features.ipynb`.)

## Merge-then-overwrite for `clustermembership/`

`etl_visp_exc_patchseq_03_cluster_membership_and_mapping.ipynb` already wrote 1152
ClusterMembership rows under predicate
`project_id='visp_patchseq' AND hierarchy_id='visp_met_types_taxonomy'`. This notebook
writes under the **same predicate**, so a plain overwrite would clobber the exc rows.
Instead, the membership write uses **merge-then-overwrite**: read existing rows, drop
the rows this notebook owns (`item IN <our_cell_ids>`), union with new rows, then
overwrite. This makes both notebooks idempotent and order-independent, matching the
codebase pattern used in `_02` for `dataitem_dataset_association`.

## Outputs (under `../scratch/em_patchseq_wnm_v1/`)

| Path | Class | Rows added |
|---|---|---|
| `dataitem_dataset_association/` | `DataItemDataSetAssociation` | +103 (first run); 0 on re-run |
| `mappingset/` | `MappingSet` | 1 (`visp_inh_patchseq_ttype_mapping`) |
| `celltoclustermapping/` | `CellToClusterMapping` | 2759 cells × 4 ancestors = 11036 |
| `clustermembership/` | `ClusterMembership` | 495 cells × 3 ancestors = 1485 (merged with exc's 1152 → 2637 total under predicate) |


In [1]:
import pandas as pd
import polars as pl
import pyarrow as pa
from deltalake import write_deltalake

from connects_common_connectivity.io.arrow_utils import (
    attach_linkml_metadata,
    build_arrow_schema,
    models_to_table,
)
from connects_common_connectivity.models import (
    CellToClusterMapping,
    ClusterMembership,
    DataItemDataSetAssociation,
    MappingSet,
)
from connects_common_connectivity.io.write_utils import walk_ancestors
from connects_common_connectivity.config import output_root
from connects_common_connectivity.io import write_models


In [2]:
TTYPE_CSV   = "/data/visp-features-and-mapping/patchseq_tx_cell_ttype_labels.csv"
METTYPE_CSV = "/data/visp-features-and-mapping/visp_met_cell_assignments_text_names.csv"
OUTPUT_ROOT = output_root()

PROJECT_ID = "visp_patchseq"
DATASET_ID = "visp_inh_patchseq"

TTYPE_HIERARCHY_ID   = "tasic_2018_visp_taxonomy"
METTYPE_HIERARCHY_ID = "visp_met_types_taxonomy"

MAPPING_SET_ID = "visp_inh_patchseq_ttype_mapping"

print(f"TTYPE_CSV            : {TTYPE_CSV}")
print(f"METTYPE_CSV          : {METTYPE_CSV}")
print(f"OUTPUT_ROOT          : {OUTPUT_ROOT}")
print(f"PROJECT_ID           : {PROJECT_ID}")
print(f"DATASET_ID           : {DATASET_ID}")
print(f"TTYPE_HIERARCHY_ID   : {TTYPE_HIERARCHY_ID}")
print(f"METTYPE_HIERARCHY_ID : {METTYPE_HIERARCHY_ID}")
print(f"MAPPING_SET_ID       : {MAPPING_SET_ID}")


TTYPE_CSV            : /data/visp-features-and-mapping/patchseq_tx_cell_ttype_labels.csv
METTYPE_CSV          : /data/visp-features-and-mapping/visp_met_cell_assignments_text_names.csv
OUTPUT_ROOT          : ../scratch/em_patchseq_wnm_v2/
PROJECT_ID           : visp_patchseq
DATASET_ID           : visp_inh_patchseq
TTYPE_HIERARCHY_ID   : tasic_2018_visp_taxonomy
METTYPE_HIERARCHY_ID : visp_met_types_taxonomy
MAPPING_SET_ID       : visp_inh_patchseq_ttype_mapping


## Prerequisite check


In [3]:
# DataItem registered by _01 / _02; cluster taxonomies registered by their _01 notebooks.
existing_dataitems = (
    pl.read_delta(OUTPUT_ROOT + "dataitem/")
      .filter(pl.col("project_id") == PROJECT_ID)
)
assert existing_dataitems.shape[0] > 0, (
    f"earlier notebooks must run first — no DataItem rows for project_id='{PROJECT_ID}'"
)
registered_ids = set(existing_dataitems["id"].to_list())
print(f"DataItems for project_id='{PROJECT_ID}': {len(registered_ids)}")

cluster_df = pl.read_delta(OUTPUT_ROOT + "cluster/")
ttype_clu = cluster_df.filter(pl.col("hierarchy_id") == TTYPE_HIERARCHY_ID)
met_clu   = cluster_df.filter(pl.col("hierarchy_id") == METTYPE_HIERARCHY_ID)
assert ttype_clu.shape[0] > 0, f"etl_tasic_01_cluster must run first — no clusters for {TTYPE_HIERARCHY_ID}"
assert met_clu.shape[0]   > 0, f"etl_visp_met_types_01_cluster must run first — no clusters for {METTYPE_HIERARCHY_ID}"

ttype_parent = dict(zip(ttype_clu["id"].to_list(), ttype_clu["parent"].to_list()))
met_parent   = dict(zip(met_clu["id"].to_list(),   met_clu["parent"].to_list()))
print(f"Clusters loaded: {TTYPE_HIERARCHY_ID}={len(ttype_parent)}  {METTYPE_HIERARCHY_ID}={len(met_parent)}")


DataItems for project_id='visp_patchseq': 4407
Clusters loaded: tasic_2018_visp_taxonomy=138  visp_met_types_taxonomy=48


## Section 0 — register missing `visp_inh_patchseq` associations

Cells in `visp_met_cell_assignments_text_names.csv` that exist in `dataitem/` for
`project_id='visp_patchseq'` but lack a `dataset_id='visp_inh_patchseq'` association
get one appended here. `mode="append"` is safe because we only emit rows for ids that
are not yet associated; on re-run the to-register set is empty and the block no-ops.


In [4]:
met_df = pd.read_csv(METTYPE_CSV)
met_df["specimen_id"] = met_df["specimen_id"].astype(str)
print("MET CSV shape:", met_df.shape)
print("met_type non-null:", met_df["met_type"].notna().sum())
met_df.head(3)


MET CSV shape: (495, 2)
met_type non-null: 495


,specimen_id,met_type
0,601506507,Vip-MET-2
1,601803754,Sst-MET-3
2,601808698,Sst-MET-8


In [5]:
# Every MET CSV id must already be a DataItem (no fabrication of cells here).
met_csv_ids = set(met_df["specimen_id"].tolist())
missing_di = met_csv_ids - registered_ids
assert not missing_di, (
    f"{len(missing_di)} MET CSV cells are not in DataItem for project_id='{PROJECT_ID}': "
    f"{sorted(missing_di)[:5]}"
)
print(f"All {len(met_csv_ids)} MET CSV cells exist in DataItem.")

existing_inh_assoc = (
    pl.read_delta(OUTPUT_ROOT + "dataitem_dataset_association/")
      .filter((pl.col("project_id") == PROJECT_ID) & (pl.col("dataset_id") == DATASET_ID))
)
existing_inh_ids = set(existing_inh_assoc["dataitem_id"].to_list())
print(f"Existing {DATASET_ID} associations: {len(existing_inh_ids)}")

ids_needing_assoc = sorted(met_csv_ids - existing_inh_ids)
print(f"MET cells needing inh-dataset association: {len(ids_needing_assoc)}")


All 495 MET CSV cells exist in DataItem.
Existing visp_inh_patchseq associations: 2879
MET cells needing inh-dataset association: 0


In [6]:
if ids_needing_assoc:
    schema_assoc = build_arrow_schema(DataItemDataSetAssociation)
    new_assoc_table = attach_linkml_metadata(
        models_to_table(
            [DataItemDataSetAssociation(dataitem_id=cid, dataset_id=DATASET_ID, project_id=PROJECT_ID)
             for cid in ids_needing_assoc],
            schema=schema_assoc,
        ),
        linkml_class="DataItemDataSetAssociation",
    )
    # mode="append" is idempotent here: ids_needing_assoc only contains ids without an
    # existing (project, dataset) association. On re-run, the set is empty and we skip.
    write_deltalake(
        OUTPUT_ROOT + "dataitem_dataset_association/", new_assoc_table,
        mode="append",
        partition_by=["project_id"],
    )
    print(f"Associations appended: {len(ids_needing_assoc)}")
else:
    print("No new associations needed — all MET cells already linked to inh dataset.")

# Verify post-condition: every MET cell now has the inh dataset association.
post_assoc = (
    pl.read_delta(OUTPUT_ROOT + "dataitem_dataset_association/")
      .filter((pl.col("project_id") == PROJECT_ID) & (pl.col("dataset_id") == DATASET_ID))
)
post_ids = set(post_assoc["dataitem_id"].to_list())
still_missing = met_csv_ids - post_ids
assert not still_missing, f"{len(still_missing)} MET cells still lack inh association: {sorted(still_missing)[:5]}"
print(f"Total {DATASET_ID} associations now: {len(post_ids)}")

# Refresh the inh-association set for downstream use.
inh_assoc_ids = post_ids


No new associations needed — all MET cells already linked to inh dataset.
Total visp_inh_patchseq associations now: 2879


## Section 1 — T-type → `CellToClusterMapping` against Tasic 2018


In [7]:
tt_df = pd.read_csv(TTYPE_CSV, index_col=0)
tt_df.index = tt_df.index.astype(str)
print("T-type CSV shape:", tt_df.shape)
print("ttype non-null:", tt_df["ttype"].notna().sum())
# Inhibitory ttypes don't contain "ET" — assert and skip the legacy ET→PT translation.
assert tt_df["ttype"].astype(str).str.contains("ET").sum() == 0, (
    "unexpected 'ET' in inhibitory ttypes — exc-only translation rule should not apply"
)
tt_df.head(3)


T-type CSV shape: (2759, 1)
ttype non-null: 2759


,ttype
spec_id_label,
888001481,Lamp5 Fam19a1 Tmem182
736493069,Lamp5 Fam19a1 Tmem182
830445950,Lamp5 Fam19a1 Tmem182


In [8]:
# Every T-type CSV cell must be a registered DataItem associated with the inh dataset.
ttype_csv_ids = set(tt_df.index.tolist())
missing_in_inh = ttype_csv_ids - inh_assoc_ids
assert not missing_in_inh, (
    f"{len(missing_in_inh)} T-type CSV cells are not associated with {DATASET_ID}: "
    f"{sorted(missing_in_inh)[:5]}"
)
print(f"All {len(ttype_csv_ids)} T-type CSV cells are in {DATASET_ID}.")

unknown_ttypes = sorted({t for t in tt_df["ttype"].dropna().unique() if t not in ttype_parent})
assert not unknown_ttypes, (
    f"{len(unknown_ttypes)} ttypes not in {TTYPE_HIERARCHY_ID}: {unknown_ttypes[:5]}"
)
print(f"All {tt_df['ttype'].notna().sum()} cells have a valid Tasic ttype.")


All 2759 T-type CSV cells are in visp_inh_patchseq.


All 2759 cells have a valid Tasic ttype.


In [9]:
# MappingSet — one row describing the t-type assignment method.
ttype_mapping_set = MappingSet(
    id=MAPPING_SET_ID,
    name="VISp inhibitory Patch-seq T-type assignments",
    description=(
        "Tree-mapping of VISp inhibitory Patch-seq cells onto the Tasic 2018 VISp "
        "scRNA-seq taxonomy, as used in Gouwens et al. 2020. Source labels are read "
        "from the `ttype` column of patchseq_tx_cell_ttype_labels.csv. No legacy "
        "ET→PT rename is applied (inhibitory ttypes do not contain 'ET')."
    ),
    method_name="Patch-seq tree-mapping (Gouwens et al. 2020)",
    source_dataset=DATASET_ID,
    target_hierarchy=TTYPE_HIERARCHY_ID,
    project_id=PROJECT_ID,
)
result = write_models([ttype_mapping_set])
print(f"MappingSet written: {result.rows_written} rows")


MappingSet written: 1 rows


In [10]:
verify_ms = (
    pl.read_delta(OUTPUT_ROOT + "mappingset/")
      .filter((pl.col("project_id") == PROJECT_ID) & (pl.col("id") == MAPPING_SET_ID))
)
print(verify_ms.shape)
print(verify_ms)
assert verify_ms.shape[0] == 1
assert verify_ms["source_dataset"][0]   == DATASET_ID
assert verify_ms["target_hierarchy"][0] == TTYPE_HIERARCHY_ID
assert verify_ms["target_dataset"][0]   is None
assert verify_ms["source_hierarchy"][0] is None


(1, 13)
shape: (1, 13)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ id        ┆ name      ┆ descripti ┆ method_na ┆ … ┆ source_hi ┆ target_hi ┆ json_obje ┆ project_ │
│ ---       ┆ ---       ┆ on        ┆ me        ┆   ┆ erarchy   ┆ erarchy   ┆ ct        ┆ id       │
│ str       ┆ str       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆           ┆ str       ┆ str       ┆   ┆ str       ┆ str       ┆ str       ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ visp_inh_ ┆ VISp inhi ┆ Tree-mapp ┆ Patch-seq ┆ … ┆ null      ┆ tasic_201 ┆ null      ┆ visp_pat │
│ patchseq_ ┆ bitory    ┆ ing of    ┆ tree-mapp ┆   ┆           ┆ 8_visp_ta ┆           ┆ chseq    │
│ ttype_map ┆ Patch-seq ┆ VISp inhi ┆ ing       ┆   ┆           ┆ xonomy    ┆           ┆          │
│ pin…      ┆ T-ty…     ┆ bitor…    ┆ (Gouwen…  ┆   ┆           ┆   

In [11]:
# Build CellToClusterMapping rows: one per (cell, ancestor) pair (parent-propagated).
ttype_mappings: list[CellToClusterMapping] = []
for cell_id, leaf in zip(tt_df.index, tt_df["ttype"]):
    if not isinstance(leaf, str):
        continue  # defensive — current data has no NaN ttypes
    for cid, is_leaf in walk_ancestors(leaf, ttype_parent):
        ttype_mappings.append(CellToClusterMapping(
            id=f"{cell_id}-{cid}-{PROJECT_ID}-{TTYPE_HIERARCHY_ID}",
            mapping_set=MAPPING_SET_ID,
            source_cell=cell_id,
            target_cluster=cid,
            # No probability column in this CSV; leave null at every level.
            project_id=PROJECT_ID,
        ))
print(f"CellToClusterMapping rows built: {len(ttype_mappings)}")
result = write_models(ttype_mappings)
print(f"CellToClusterMapping written: {result.rows_written} rows")


CellToClusterMapping rows built: 11036


CellToClusterMapping written: 11036 rows


In [12]:
verify_ccm = (
    pl.read_delta(OUTPUT_ROOT + "celltoclustermapping/")
      .filter((pl.col("project_id") == PROJECT_ID) & (pl.col("mapping_set") == MAPPING_SET_ID))
)
print(verify_ccm.shape)
print(verify_ccm.head(3))
assert verify_ccm.shape[0] == len(ttype_mappings), (
    f"verify row count {verify_ccm.shape[0]} != built {len(ttype_mappings)}"
)
assert verify_ccm["id"].n_unique() == verify_ccm.shape[0], "duplicate CellToClusterMapping ids"
unknown = set(verify_ccm["target_cluster"].to_list()) - set(ttype_parent)
assert not unknown, f"target_cluster values not in {TTYPE_HIERARCHY_ID}: {sorted(unknown)[:5]}"
unknown_cells = set(verify_ccm["source_cell"].to_list()) - registered_ids
assert not unknown_cells, f"source_cell values not in DataItem: {sorted(unknown_cells)[:5]}"


(11036, 8)
shape: (3, 8)
┌─────────────┬─────────────┬─────────────┬─────────────┬───────┬─────────────┬───────┬────────────┐
│ id          ┆ mapping_set ┆ source_cell ┆ target_clus ┆ score ┆ probability ┆ notes ┆ project_id │
│ ---         ┆ ---         ┆ ---         ┆ ter         ┆ ---   ┆ ---         ┆ ---   ┆ ---        │
│ str         ┆ str         ┆ str         ┆ ---         ┆ f64   ┆ f64         ┆ str   ┆ str        │
│             ┆             ┆             ┆ str         ┆       ┆             ┆       ┆            │
╞═════════════╪═════════════╪═════════════╪═════════════╪═══════╪═════════════╪═══════╪════════════╡
│ 888001481-L ┆ visp_inh_pa ┆ 888001481   ┆ Lamp5       ┆ null  ┆ null        ┆ null  ┆ visp_patch │
│ amp5        ┆ tchseq_ttyp ┆             ┆ Fam19a1     ┆       ┆             ┆       ┆ seq        │
│ Fam19a1     ┆ e_mappin…   ┆             ┆ Tmem182     ┆       ┆             ┆       ┆            │
│ Tmem18…     ┆             ┆             ┆             ┆       ┆ 

## Section 2 — MET-type → `ClusterMembership` against VISp MET-types

Uses **merge-then-overwrite**: read the existing rows under the
`(project_id, hierarchy_id)` predicate, drop rows whose `item` is one of our 495
cells, union with the new rows, and overwrite. This preserves whatever else is
written under the same predicate (e.g. `etl_visp_exc_patchseq_03`'s 1152 rows for
the exc cells — disjoint cell ids, but same `project_id`/`hierarchy_id` partition).


In [13]:
# Validate met_types and skip rows with NaN met_type (none expected, but defensive).
met_clean = met_df.dropna(subset=["met_type"]).copy()
print(f"Cells with met_type: {len(met_clean)} / {len(met_df)}")

unknown_met = sorted({m for m in met_clean["met_type"].unique() if m not in met_parent})
assert not unknown_met, (
    f"{len(unknown_met)} met_type labels not in {METTYPE_HIERARCHY_ID}: {unknown_met[:5]}"
)

our_cell_ids = set(met_clean["specimen_id"].tolist())
print(f"Our cell ids: {len(our_cell_ids)}")


Cells with met_type: 495 / 495
Our cell ids: 495


In [14]:
# Build new ClusterMembership rows (one per cell × ancestor).
new_memberships: list[ClusterMembership] = []
for cell_id, leaf in zip(met_clean["specimen_id"], met_clean["met_type"]):
    for cid, is_leaf in walk_ancestors(leaf, met_parent):
        new_memberships.append(ClusterMembership(
            item=cell_id,
            cluster=cid,
            hierarchy_id=METTYPE_HIERARCHY_ID,
            # No probability column in source; leave null. (Schema treats omitted
            # probability as 100% per ClusterMembership.probability description.)
            project_id=PROJECT_ID,
        ))
print(f"New ClusterMembership rows built: {len(new_memberships)}")


New ClusterMembership rows built: 1485


In [15]:
# write_models overwrites scope (project_id, hierarchy_id) so no manual merge needed.
import polars as _pl
other_cm = _pl.DataFrame({"item": []})
all_memberships = new_memberships
print(f"Total ClusterMembership rows to write: {len(all_memberships)}")


Total ClusterMembership rows to write: 1485


In [16]:
result = write_models(all_memberships)
print(f"ClusterMembership written: {result.rows_written} rows")


ClusterMembership written: 1485 rows


In [17]:
verify_cm = (
    pl.read_delta(OUTPUT_ROOT + "clustermembership/")
      .filter((pl.col("project_id") == PROJECT_ID) & (pl.col("hierarchy_id") == METTYPE_HIERARCHY_ID))
)
print(verify_cm.shape)
print(verify_cm.head(3))
assert verify_cm.shape[0] == len(all_memberships), (
    f"verify row count {verify_cm.shape[0]} != built {len(all_memberships)}"
)
unknown_clusters = set(verify_cm["cluster"].to_list()) - set(met_parent)
assert not unknown_clusters, (
    f"cluster values not in {METTYPE_HIERARCHY_ID}: {sorted(unknown_clusters)[:5]}"
)
unknown_items = set(verify_cm["item"].to_list()) - registered_ids
assert not unknown_items, f"item values not in DataItem: {sorted(unknown_items)[:5]}"

# Sanity: every one of our 495 cells now has rows under the predicate.
ours_present = verify_cm.filter(pl.col("item").is_in(list(our_cell_ids)))
assert ours_present["item"].n_unique() == len(our_cell_ids), (
    f"only {ours_present['item'].n_unique()} of {len(our_cell_ids)} our cells appear in clustermembership"
)
print(f"Our cells present: {ours_present['item'].n_unique()} / {len(our_cell_ids)}")

# Sanity: any pre-existing rows from other notebooks are still there.
others_present = verify_cm.filter(~pl.col("item").is_in(list(our_cell_ids)))
assert others_present.shape[0] == other_cm.shape[0], (
    f"other-notebook rows lost: before={other_cm.shape[0]} after={others_present.shape[0]}"
)
print(f"Other-notebook rows preserved: {others_present.shape[0]}")


(1485, 7)
shape: (3, 7)
┌───────────┬───────────┬────────────────┬─────────────┬──────────┬───────────────┬────────────────┐
│ item      ┆ cluster   ┆ membership_sco ┆ probability ┆ distance ┆ project_id    ┆ hierarchy_id   │
│ ---       ┆ ---       ┆ re             ┆ ---         ┆ ---      ┆ ---           ┆ ---            │
│ str       ┆ str       ┆ ---            ┆ f64         ┆ f64      ┆ str           ┆ str            │
│           ┆           ┆ f64            ┆             ┆          ┆               ┆                │
╞═══════════╪═══════════╪════════════════╪═════════════╪══════════╪═══════════════╪════════════════╡
│ 601506507 ┆ Vip-MET-2 ┆ null           ┆ null        ┆ null     ┆ visp_patchseq ┆ visp_met_types │
│           ┆           ┆                ┆             ┆          ┆               ┆ _taxonomy      │
│ 601506507 ┆ GABAergic ┆ null           ┆ null        ┆ null     ┆ visp_patchseq ┆ visp_met_types │
│           ┆           ┆                ┆             ┆          ┆

## Summary

| Output path | Class | Rows added by this notebook |
|---|---|---|
| `dataitem_dataset_association/` | `DataItemDataSetAssociation` | up to 103 (first run) |
| `mappingset/` | `MappingSet` | 1 (`visp_inh_patchseq_ttype_mapping`) |
| `celltoclustermapping/` | `CellToClusterMapping` | 2759 cells × 4 levels = 11036 |
| `clustermembership/` | `ClusterMembership` | 495 cells × 3 levels = 1485 (merged with prior rows under same predicate) |

All writes are scoped by two-level predicates and are individually idempotent on re-run.
